In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project="prod-organize-arizon-4e1c0a83")

query = """
SELECT
*
FROM `prod-organize-arizon-4e1c0a83.viewers_dataset.az_pct_voters_2024`
"""

df = client.query(query).to_dataframe()

In [ ]:
df.head()

,pctnum,population_count,dem_votes,rep_votes,dem_margin,third_votes,GEOMETRY
0,MO0220,111412,5932,27805,0.213343,11298,"POLYGON((-113.665315 34.212622, -113.633412 34..."
1,MC0212,7981,847,2607,0.324895,1467,"POLYGON((-111.89062 33.698686, -111.890629 33...."
2,MC0599,16184,2345,1927,1.216917,2922,"POLYGON((-111.943367 33.480195, -111.934764 33..."
3,MC0078,9402,1166,2869,0.406413,1736,"POLYGON((-111.890597 33.726735, -111.925592 33..."
4,MC0171,9755,1124,2995,0.375292,1834,"POLYGON((-111.669425 33.277983, -111.68625 33...."


In [ ]:
# have to create geometry that folium can work with from bigquery friendly wkt geometry field
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.validation import make_valid  # Shapely ≥2.0

# point to the column that contains WKT, here it's called GEOMETRY
wkt_col = "GEOMETRY"

# parse WKT into shapely geometries
geoms = gpd.GeoSeries.from_wkt(df[wkt_col])

# build a GeoDataFrame and assign the correct CRS
# If WKT is already lon/lat (WGS84), use EPSG:4326. Otherwise, set the true source EPSG and then .to_crs(4326).
gdf = gpd.GeoDataFrame(df.drop(columns=["geometry"], errors="ignore"),
                       geometry=geoms,
                       crs="EPSG:4326")


In [ ]:
import folium
import numpy as np
import pandas as pd
from branca.colormap import LinearColormap
from folium.features import GeoJson, GeoJsonTooltip

# ============================================
KEY_COL  = "pctnum"
POP_COL  = "population_count"
DEM_COL  = "dem_votes"
REP_COL  = "rep_votes"
OTH_COL  = "third_votes"
MARGIN_COL = "dem_margin"
PCT_NAME = "PRECINCTNA"
# ============================================

# Re-compute Dem margin as TWO-PARTY share centered at 0: (D - R) / (D + R)
# This gives values in [-1, +1]; blue = negative (more Dem), red = positive (more Rep).
twoparty = gdf[[DEM_COL, REP_COL]].sum(axis=1, skipna=True)
with np.errstate(divide='ignore', invalid='ignore'):
    gdf["dem_margin_2p"] = np.where(twoparty > 0,
                                    (gdf[REP_COL] - gdf[DEM_COL]) / twoparty,
                                    np.nan)


minx, miny, maxx, maxy = gdf.total_bounds
m = folium.Map(location=[(miny+maxy)/2, (minx+maxx)/2], zoom_start=6, tiles="CartoDB Positron")

abs_margin = gdf["dem_margin_2p"].abs().dropna()
vmax = float(np.quantile(abs_margin, 0.98)) if len(abs_margin) else 1.0
cmap = LinearColormap(colors=["#2166ac", "#f7f7f7", "#b2182b"], vmin=-vmax, vmax=vmax)
cmap.caption = "Rep margin (two-party, Dem − Rep)"
cmap.add_to(m)


def style_fn(feat):
    val = feat["properties"].get("dem_margin_2p")
    if pd.isna(val):
        return {"fillColor": "#cccccc", "fillOpacity": 0.25, "weight": 0.4, "color": "#666"}
    return {"fillColor": cmap(val), "fillOpacity": 0.8, "weight": 0.4, "color": "#666"}

tooltip_fields = [c for c in [KEY_COL, "dem_margin_2p", POP_COL] if c in gdf.columns]
tooltip_aliases = ["Pct:", "Rep margin (2-party):", "Population:"]

poly_layer = GeoJson(
    data=gdf.to_json(),
    name="Precincts — Rep margin",
    style_function=style_fn,
    highlight_function=lambda f: {"weight": 2, "color": "#000"},
    tooltip=GeoJsonTooltip(fields=tooltip_fields, aliases=tooltip_aliases, localize=True, sticky=False),
)
poly_layer.add_to(m)

# Population “spikes” as scaled circle markers at representative points
# Population “spikes” as scaled circle markers at representative points
pop_layer = folium.FeatureGroup(name="Population spikes", show=False)

max_pop = gdf[POP_COL].max() if POP_COL in gdf.columns else None

def pop_radius(pop, min_r=3, max_r=18):
    if (pop is None) or pd.isna(pop) or (max_pop is None) or (max_pop <= 0):
        return 0
    return float(min_r + (max_r - min_r) * np.sqrt(pop / max_pop))

# --- SAFER: filter to valid geometries, compute points, drop null points & pops
gdf_pts = gdf[gdf.geometry.notna() & ~gdf.geometry.apply(lambda g: getattr(g, "is_empty", True))].copy()
gdf_pts["__pt"] = gdf_pts.geometry.apply(lambda g: g.representative_point() if (g is not None and not g.is_empty) else None)
gdf_pts = gdf_pts[gdf_pts["__pt"].notna() & gdf_pts[POP_COL].notna()].copy()

for _, row in gdf_pts.iterrows():
    r = pop_radius(row[POP_COL])
    if r <= 0:
        continue
    lat, lon = float(row["__pt"].y), float(row["__pt"].x)
    folium.CircleMarker(
        location=[lat, lon],
        radius=r,
        weight=0.8,
        color="#222",
        fill=True,
        fill_opacity=0.45,
        fill_color="#ffffff",
        popup=folium.Popup(
            f"Pct: {row.get(KEY_COL,'')}"
            + (f"<br>Population: {int(row[POP_COL]):,}" if pd.notna(row[POP_COL]) else "")
            + (f"<br>Dem margin (2p): {row['dem_margin_for_map']:.3f}" if pd.notna(row.get('dem_margin_for_map')) else ""),
            max_width=320
        ),
    ).add_to(pop_layer)

pop_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny, minx], [maxy, maxx]])
m.save("az_precincts_demmargin_popspikes.html")

In [ ]:
from IPython.display import IFrame
IFrame("map.html", width="100%", height=600)

m


In [ ]:
from google.cloud import storage

client = storage.Client(project="prod-organize-arizon-4e1c0a83")
bucket = client.bucket("cmarikos-maps")
blob = bucket.blob("maps/current_map.html")
blob.upload_from_filename("az_precincts_demmargin_popspikes.html")

print("Public URL:", f"https://storage.googleapis.com/{bucket.name}/{blob.name}")


Public URL: https://storage.googleapis.com/cmarikos-maps/maps/current_map.html
